In [ ]:
import torch.nn as nn
import torch
from torch.nn.utils.rnn import pack_padded_sequence

In [62]:
# Каркас LSTM-классификатора: сохраните общий интерфейс RNNTextClassifier.
class LSTMTextClassifier(nn.Module):
    """Классификатор AG News на LSTM."""

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        pad_idx: int,
        embedding_dim: int = 128,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
        bidirectional: bool = True,
    ) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_directions = 2 if bidirectional else 1

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # Создайте nn.LSTM с input_size, hidden_size, num_layers,
        # batch_first=True, bidirectional=bidirectional и dropout при num_layers > 1.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * self.num_directions, num_classes)

    def forward(self, tokens: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().to("cpu"),
            batch_first=True,
            enforce_sorted=False,
        )

        # Получите h_n и c_n из self.lstm(packed).
        # Для классификации используйте hidden; cell нужен самой LSTM для обновления памяти.
        _, (hidden, cell) = self.lstm(packed)

        # Приведите hidden к форме [num_layers, directions, B, hidden_size].
        # Затем выберите последний слой и объедините направления в [B, directions * hidden_size].
        hidden = hidden.view(
            self.num_layers,
            self.num_directions,
            tokens.size(0),
            self.hidden_size,
        )
        last_layer = hidden[-1]
        document_vector = last_layer.transpose(0, 1).reshape(tokens.size(0), -1)

        return self.classifier(self.dropout(document_vector))

In [63]:
# Каркас GRU-классификатора: заполните создание слоя и извлечение итогового hidden state.
class GRUTextClassifier(nn.Module):
    """Классификатор AG News на GRU."""

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        pad_idx: int,
        embedding_dim: int = 128,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
        bidirectional: bool = True,
    ) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_directions = 2 if bidirectional else 1

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # Создайте nn.GRU с теми же параметрами, что и nn.RNN/nn.LSTM.
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * self.num_directions, num_classes)

    def forward(self, tokens: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().to("cpu"),
            batch_first=True,
            enforce_sorted=False,
        )

        # Получите hidden из self.gru(packed).
        _, hidden = self.gru(packed)

        # Преобразуйте hidden к форме [num_layers, directions, B, hidden_size],
        # выберите последний слой и объедините направления.
        hidden = hidden.view(
            self.num_layers,
            self.num_directions,
            tokens.size(0),
            self.hidden_size,
        )
        last_layer = hidden[-1]
        document_vector = last_layer.transpose(0, 1).reshape(tokens.size(0), -1)

        return self.classifier(self.dropout(document_vector))